# Compare results of SWITi and inner tiling

## Gradient Test

#### Load Images

In [ ]:
import numpy as np
import os
import ndv
import tifffile as tiff

In [ ]:
DATASET = "PaviaATN"
IMG_IDX = 0

In [ ]:
root_dir = f"/project/careamics/switi/results/{DATASET}/predictions"

In [ ]:
inner_tiling_data = np.load(os.path.join(root_dir, "inner_tiling", "predictions.npz"), allow_pickle=True)
switi_data = np.load(os.path.join(root_dir, "sw_inner_tiling", "predictions.npz"), allow_pickle=True)

print(switi_data.files, inner_tiling_data.files)

FNAME = switi_data.files[IMG_IDX]
switi_img = switi_data[FNAME].squeeze()
inner_tiling_img = inner_tiling_data[FNAME].squeeze()
gt_img = tiff.imread(f"/project/careamics/switi/data/{DATASET}/targets/test/{FNAME}.tif")

switi_img.shape, inner_tiling_img.shape, gt_img.shape

In [ ]:
v1 = ndv.imshow(switi_img)

In [ ]:
v2 = ndv.imshow(inner_tiling_img)

In [ ]:
v3 = ndv.imshow(gt_img)

#### Check gradient maps

In [ ]:
from analysis_pipeline.gradient_test.gradient_analysis import compute_gradients
from analysis_pipeline.gradient_test.plotting import plot_gradient_comparison

In [ ]:
CHANNEL = 0

In [ ]:
inner_tiling_grads = compute_gradients(inner_tiling_img[CHANNEL])
switi_grads = compute_gradients(switi_img[CHANNEL])
gt_grads = compute_gradients(gt_img[CHANNEL])
[grad.shape for grad in inner_tiling_grads], [grad.shape for grad in switi_grads], [grad.shape for grad in gt_grads]

In [ ]:
fig = plot_gradient_comparison(
    {
        "Inner Tiling": inner_tiling_grads,
        "SWiTi": switi_grads,
        "Ground Truth": gt_grads
    },
    y_ROI=(2416, 2593), #(150, 300), #(1300, 1600) # (1440,2416 1540)
    x_ROI=(0, 177) #(1450, 1600), #(1280, 1580) # (1310, 1410)
)

#### Run the gradient tests

In [ ]:
from analysis_pipeline.gradient_test.analysis import run_gradient_analysis

In [ ]:
inner_tiling_reports = []
for i in range(inner_tiling_img.shape[0]):
    report = run_gradient_analysis(
        inner_tiling_img[np.newaxis, ...],
        tile_size=[64, 64],
        overlap=[32, 32],
        channel=i,
        statistic="js",
        save_dir=None,
        image_names=[f"img{i}"]
    )
    inner_tiling_reports.append(report)

In [ ]:
switi_reports = []
for i in range(switi_img.shape[0]):
    report = run_gradient_analysis(
        switi_img[np.newaxis, ...],
        tile_size=[64, 64],
        overlap=[32, 32],
        channel=i,
        statistic="js",
        save_dir=None,
        image_names=[f"img{i}"]
    )
    switi_reports.append(report)

In [ ]:
gt_reports = []
for i in range(gt_img.shape[0]):
    report = run_gradient_analysis(
        gt_img[np.newaxis, ...],
        tile_size=[64, 64],
        overlap=[32, 32],
        channel=i,
        statistic="js",
        save_dir=None,
        image_names=[f"img{i}"]
    )
    gt_reports.append(report)

#### Plot results of the gradient tests

In [ ]:
from analysis_pipeline.gradient_test.plotting import (
    plot_pvalue_distribution,
    plot_significance_overlay,
    plot_significance_overlay_grid
)

In [ ]:
fig = plot_significance_overlay(
    [switi_reports[i].images[0] for i in range(len(switi_reports))],
    switi_img,
    channels=list(range(switi_img.shape[0])),
    tile_size=[64, 64],
    overlap=[32, 32],
    title="SWITi",
    overlay_alpha=0.5,
    y_ROI=(1440, 1540),# (150, 300),# (1300, 1600) # (1440, 1540)
    x_ROI=(1310, 1410),# (1310, 1410) #(1280, 1580) # (1310, 1410)
)

In [ ]:
fig = plot_significance_overlay(
    [inner_tiling_reports[i].images[0] for i in range(len(inner_tiling_reports))],
    inner_tiling_img,
    channels=list(range(inner_tiling_img.shape[0])),
    tile_size=[64, 64],
    overlap=[32, 32],
    title="Inner Tiling",
    overlay_alpha=0.5,
    y_ROI=(1300, 1600),# (150, 300),# (1300, 1600) # (1440, 1540)
    x_ROI=(1280, 1580),# (1310, 1410) #(1280, 1580) # (1310, 1410)
)

In [ ]:
fig = plot_significance_overlay(
    [gt_reports[i].images[0] for i in range(len(gt_reports))],
    gt_img,
    channels=list(range(gt_img.shape[0])),
    tile_size=[64, 64],
    overlap=[32, 32],
    title="Ground Truth",
    overlay_alpha=0.5,
    y_ROI=(1300, 1600),# (150, 300),# (1300, 1600) # (1440, 1540)
    x_ROI=(1280, 1580),# (1310, 1410) #(1280, 1580) # (1310, 1410)
)

#### Check results of gradient test in flat regions

SWITi shows significant divergence in flat regions. That's probably because also small discrepancies sticks out in flat regions, as there is no natural variability in the image and so also a tiny gradient can be significant.

In [ ]:
inner_tiling_report_js = run_gradient_analysis(
    inner_tiling_img[np.newaxis, :, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    channel=0,
    statistic="js",
    save_dir=None
)

In [ ]:
fig = plot_significance_overlay(
    inner_tiling_report_js.images[0],
    inner_tiling_img[0, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    title="Inner Tiling",
)

In [ ]:
switi_report_js = run_gradient_analysis(
    switi_img[np.newaxis, :, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    channel=0,
    statistic="js",
    save_dir=None
)

In [ ]:
fig = plot_significance_overlay(
    switi_report_js.images[0],
    switi_img[0, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    title="SWITi",
)

In [ ]:
gt_report_js = run_gradient_analysis(
    gt_img[np.newaxis, :, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    channel=0,
    statistic="js",
    save_dir=None
)

In [ ]:
fig = plot_significance_overlay(
    gt_report_js.images[0],
    gt_img[0, 2416:2593, 0:177],
    tile_size=[64, 64],
    overlap=[32, 32],
    title="SWITi",
)

## FRC metric

#### Load images

In [ ]:
import numpy as np
import os
import ndv
import tifffile as tiff

In [ ]:
DATASET = "PaviaATN"
IMG_IDX = 0

In [ ]:
root_dir = f"/project/careamics/switi/results/{DATASET}/predictions"

In [ ]:
inner_tiling_data = np.load(os.path.join(root_dir, "inner_tiling", "predictions.npz"), allow_pickle=True)
switi_data = np.load(os.path.join(root_dir, "sw_inner_tiling", "predictions.npz"), allow_pickle=True)

print(switi_data.files, inner_tiling_data.files)

FNAME = switi_data.files[IMG_IDX]
switi_img = switi_data[FNAME].squeeze()
inner_tiling_img = inner_tiling_data[FNAME].squeeze()
gt_img = tiff.imread(f"/project/careamics/switi/data/{DATASET}/targets/test/{FNAME}.tif")

switi_img.shape, inner_tiling_img.shape, gt_img.shape

In [ ]:
v1 = ndv.imshow(switi_img)

In [ ]:
v2 = ndv.imshow(inner_tiling_img)

In [ ]:
v3 = ndv.imshow(gt_img)

#### Run FRC metric on images

In [ ]:
from analysis_pipeline.frc.analysis import run_frc_analysis

In [ ]:
frc_report_switi = run_frc_analysis(
    switi_img[np.newaxis, ...],
    gt_img[np.newaxis, ...],
    save_dir=None,
    method_name="SWITi",
    channel=0
)

In [ ]:
frc_report_inner_tiling = run_frc_analysis(
    inner_tiling_img[np.newaxis, ...],
    gt_img[np.newaxis, ...],
    save_dir=None,
    method_name="inner_tiling",
    channel=0
)

In [ ]:
frc_report_switi